In [1]:
import requests
import zipfile
import io
from pathlib import Path

# -------------------------
# CONFIGURATION
# -------------------------
START_YEAR = 2009
END_YEAR = 2014
TARGET_DIR = Path('/scratch/ng72/ms5578/mmsdm_backup')  # folder to save DISPATCHLOAD ZIPs
TEMP_DIR = Path("/scratch/ng72/ms5578/temp_monthly_zips")    # temporary folder for monthly ZIPs
TIMEOUT = 20
# -------------------------

TARGET_DIR.mkdir(exist_ok=True)
TEMP_DIR.mkdir(exist_ok=True)

def download_monthly_zip(year, month):
    """Download the full monthly ZIP file."""
    url = f"https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/{year}/MMSDM_{year}_{month:02d}.zip"
    temp_zip_path = TEMP_DIR / f"MMSDM_{year}_{month:02d}.zip"

    if temp_zip_path.exists():
        print(f"✓ Monthly ZIP already exists: {temp_zip_path.name}")
        return temp_zip_path

    print(f"→ Downloading monthly ZIP: {temp_zip_path.name}")
    try:
        r = requests.get(url, timeout=TIMEOUT)
        if r.status_code == 200:
            temp_zip_path.write_bytes(r.content)
            print(f"✓ Saved monthly ZIP: {temp_zip_path.name}")
            return temp_zip_path
        else:
            print(f"✗ Missing monthly ZIP: {url} (HTTP {r.status_code})")
            return None
    except Exception as e:
        print(f"⚠ Error downloading {url}: {e}")
        return None

def extract_dispatchload(monthly_zip_path, year, month):
    """Extract only the DISPATCHLOAD ZIP inside the monthly ZIP."""
    ym = f"{year}{month:02d}"
    try:
        with zipfile.ZipFile(monthly_zip_path, 'r') as z:
            # Look for the inner DISPATCHLOAD ZIP
            dispatch_files = [f for f in z.namelist() if "DISPATCHLOAD" in f and f.endswith(".zip")]
            if not dispatch_files:
                print(f"✗ No DISPATCHLOAD found in {monthly_zip_path.name}")
                return
            for f in dispatch_files:
                data = z.read(f)
                out_path = TARGET_DIR / Path(f).name
                out_path.write_bytes(data)
                print(f"✓ Extracted DISPATCHLOAD: {out_path.name}")
    except zipfile.BadZipFile:
        print(f"⚠ Bad ZIP file: {monthly_zip_path.name}")
    except Exception as e:
        print(f"⚠ Error extracting {monthly_zip_path.name}: {e}")

# -------------------------
# MAIN LOOP
# -------------------------
for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        monthly_zip = download_monthly_zip(year, month)
        if monthly_zip:
            extract_dispatchload(monthly_zip, year, month)

print("✅ Done. All DISPATCHLOAD ZIPs are in:", TARGET_DIR.resolve())


→ Downloading monthly ZIP: MMSDM_2009_01.zip
✗ Missing monthly ZIP: https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/2009/MMSDM_2009_01.zip (HTTP 404)
→ Downloading monthly ZIP: MMSDM_2009_02.zip
✗ Missing monthly ZIP: https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/2009/MMSDM_2009_02.zip (HTTP 404)
→ Downloading monthly ZIP: MMSDM_2009_03.zip
✗ Missing monthly ZIP: https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/2009/MMSDM_2009_03.zip (HTTP 404)
→ Downloading monthly ZIP: MMSDM_2009_04.zip
✗ Missing monthly ZIP: https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/2009/MMSDM_2009_04.zip (HTTP 404)
→ Downloading monthly ZIP: MMSDM_2009_05.zip
✗ Missing monthly ZIP: https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/2009/MMSDM_2009_05.zip (HTTP 404)
→ Downloading monthly ZIP: MMSDM_2009_06.zip
✗ Missing monthly ZIP: https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/2009/MMSDM_2009_06.z

KeyboardInterrupt: 